In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from fuzzywuzzy import process

/Users/idasom/opt/anaconda3/envs/myvenv/lib/python3.9/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [52]:
ratings_data = pd.read_csv("./data/the_movie_dataset/ratings_small.csv", encoding='cp949')
ratings_data = ratings_data.drop('timestamp', axis = 1)
ratings_data.head()

,userId,movieId,rating
0,1,31,2.5
1,1,1029,3.0
2,1,1061,3.0
3,1,1129,2.0
4,1,1172,4.0


In [53]:
movie_names = pd.read_csv("./data/the_movie_data_addKeyWord.csv",encoding='cp949')
movie_names = movie_names[['title', 'id']]
movie_names.head()

,title,id
0,Toy Story,862
1,Jumanji,8844
2,Grumpier Old Men,15602
3,Waiting to Exhale,31357
4,Father of the Bride Part II,11862


In [54]:
movie_names.dtypes

title    object
id        int64
dtype: object

In [55]:
movie_names.astype({'id':'int64'})

,title,id
0,Toy Story,862
1,Jumanji,8844
2,Grumpier Old Men,15602
3,Waiting to Exhale,31357
4,Father of the Bride Part II,11862
...,...,...
45458,Subdue,439050
45459,Century of Birthing,111109
45460,Betrayal,67758
45461,Satan Triumphant,227506


In [56]:
movie_names.columns = ['title', 'movieId']

In [57]:
movie_data=pd.merge(left = ratings_data , right = movie_names, how = "inner", on = "movieId")

In [58]:
#title별로 rating의 mean값을 구하자.
#또한 이때의 rating의 개수를 출력한다. 
trend = pd.DataFrame(movie_data.groupby('movieId')['rating'].mean())
trend['total number of ratings'] = pd.DataFrame(movie_data.groupby('movieId')['rating'].count()) 

trend.head(20)

,rating,total number of ratings
movieId,,
2,3.401869,107
3,3.161017,59
5,3.267857,56
6,3.884615,104
11,3.689024,82
12,2.861111,18
13,3.937500,8
14,3.451613,31
15,2.318182,11


In [59]:
# rating의 평균을 구하고, 평균 점수가 높은 영화를 출력한다. 
movie_data.groupby('movieId')['rating'].mean().sort_values(ascending=True).head(10)

movieId
66659    0.5
2191     0.5
27376    0.5
66066    0.5
27857    0.5
7093     0.5
39408    0.5
54768    0.5
80350    0.5
8963     0.5
Name: rating, dtype: float64

In [63]:
print(ratings_data.groupby('movieId')['movieId'].count())

movieId
1         247
2         107
3          59
4          13
5          56
         ... 
161944      1
162376      1
162542      1
162672      1
163949      1
Name: movieId, Length: 9066, dtype: int64


In [61]:
print(ratings_data['movieId'].count)

<bound method Series.count of 0           31
1         1029
2         1061
3         1129
4         1172
          ... 
99999     6268
100000    6269
100001    6365
100002    6385
100003    6565
Name: movieId, Length: 100004, dtype: int64>


In [13]:
# Item - based collaborative filtering 
movies_users = ratings_data.pivot_table(index=['movieId'], columns=['userId'], values='rating').fillna(0)
movies_users

userId,1,2,3,4,5,6,7,8,9,10,...,662,663,664,665,666,667,668,669,670,671
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,4.0,0.0,...,0.0,4.0,3.5,0.0,0.0,0.0,0.0,0.0,4.0,5.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161944,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
162376,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
162542,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# 우리의 최종 데이터셋은 671 x 9066이다. 
# 심지어 이 예제에서는 small dataset을 사용했다. 실제 데이터를 사용하면 더 커질 것이다.
# 희소행렬(행렬의 대부분이 0인 행렬)을 줄이기 위해서 scipy라이브러리를 사용하자.
from scipy.sparse import csr_matrix
mat_movies_users=csr_matrix(movies_users.values)
mat_movies_users

<9066x671 sparse matrix of type '<class 'numpy.float64'>'
	with 100004 stored elements in Compressed Sparse Row format>

In [15]:
# Cosine 유사도를 이용한 KNN 
# 데이터가 다차원 희소 행렬이므로 코사인 유사도를 사용한다.
model_knn= NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20, n_jobs=-1)

In [45]:
# 기준 영화 이름 , 행렬 데이터, 사용 모델, 추천할 영화 개수 
def Recommender(movie_id, data, model, n_recommendations):
    model.fit(data)
    #영화 이름을 이용하여 movie_names라는 전역 변수 영역에 있는 데이터셋에서 영화의 index를 찾아옴 
    #movie_index = process.extractOne(movie_name, movie_names['title'])[2]
    movie_title =  movie_names[movie_names['movieId'] == movie_id] #title 한개
    #print(movie_title)
    print('Movie Selected:',movie_title['title'], ', Index: ', movie_id)
    print('Searching for recommendations.....')
    #print(data[movie_id])
    #knn을 통해서 가까운 n개의 이웃을 추출함 
    distances, indices = model.kneighbors(data[movie_id], n_neighbors=n_recommendations)
    recc_movie_indices = sorted(list(zip(indices.squeeze().tolist(),distances.squeeze().tolist())),key=lambda x: x[1])[:0:-1]
    recommend_frame = []
    for val in recc_movie_indices:
        recommend_frame.append({'Title':movie_names['title'][val[0]],'Distance':val[1]})
    
    df = pd.DataFrame(recommend_frame, index = range(1,n_recommendations))
     
    return df

In [46]:
#키워드 선정을 통하여 골라진 영화의 id를 담고 있는 배열
selected_movie = [862,62,11,680,8587]

In [47]:
a = Recommender(27205, mat_movies_users, model_knn, n_recommendations)
print(a)

Movie Selected: 15480    Inception
Name: title, dtype: object , Index:  27205
Searching for recommendations.....


IndexError: row index (27205) out of range

In [49]:
n_recommendations = 10
for i in (selected_movie):
    #print(i)
    a = Recommender(i, mat_movies_users, model_knn, n_recommendations)
    print(a)

Movie Selected: 0    Toy Story
Name: title, dtype: object , Index:  862
Searching for recommendations.....
                                        Title  Distance
1                                 Deep Impact  0.531059
2                                  Radio Days  0.522100
3               Universal Soldier: The Return  0.500547
4                             Pharaoh's  Army  0.499737
5                         Spirits of the Dead  0.484906
6                                Magic Hunter  0.470963
7  And Now for Something Completely Different  0.463323
8                                     Kingpin  0.402656
9                               Force of Evil  0.363171
Movie Selected: 897    2001: A Space Odyssey
Name: title, dtype: object , Index:  62
Searching for recommendations.....
                     Title  Distance
1          The Last Valley  0.607768
2  Exorcist: The Beginning  0.607768
3           Into the Woods  0.607768
4       Night and the City  0.607768
5     From Noon Till Three  